In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce
import pickle
from scipy import sparse


# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import sys
from pathlib import Path

_start = Path.cwd().resolve()
_repo_root = next((p for p in [_start, *_start.parents] if (p / "chronopack").exists()), _start)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import chronopack as cp
from chronopack.tl import _gene_dynamics as gd
from chronopack.preprocessing import filter_low_count_genes, filter_technical_genes


In [2]:
# using CPM @ 1e6 scale factor
adata_hvg = sc.read_h5ad('../output/adata_hvg.h5ad')
adata = sc.read_h5ad('../output/adata_full.h5ad')

# Prepare input files for TradeSeq

In [3]:
mtx_agg, cpm_ref, logcpm_ref = gd.load_negative_control_logcpm('../../thp1-k562-lps.hg38.rep/intmd/k562_neg-ctrl_mtx-agg.pkl')


In [4]:
cpm, logcpm = gd.temporal_pseudobulk_from_anndata(adata)


In [5]:
shared_genes = list(set(adata.var.index) & set(logcpm_ref.columns))

cpm_ref = cpm_ref.loc[:, shared_genes]
logcpm_ref = logcpm_ref.loc[:, shared_genes]

cpm = cpm.loc[:, shared_genes]
logcpm = logcpm.loc[:, shared_genes]

adata.shape

(4034, 18833)

In [6]:
corr = gd.compute_smooth_correlations(
    logcpm,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.8,
    local_fraction_threshold=0.5,
)


In [7]:
corr = gd.refilter_tag_bias_candidates(
    corr,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr["tag_bias_candidate"].sum()))

corr[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


Tag-bias candidates: 816


,pearson_r_residual,residual_explained_total_fraction,pearson_r_diff,diff_residual_explained_fraction,tag_bias_candidate_residual,tag_bias_candidate_diff,tag_bias_candidate
gene,,,,,,,
LINC03060,1.000000,0.499872,1.000000,0.681521,True,True,True
DDX25,1.000000,0.500095,1.000000,0.759077,True,True,True
LINC03031,1.000000,0.500012,1.000000,0.767349,True,True,True
SAMD12,1.000000,0.499928,1.000000,0.741651,True,True,True
GSCAR,1.000000,0.263736,1.000000,0.521177,True,True,True
ENSG00000300630,1.000000,0.499547,1.000000,0.751198,True,True,True
TM4SF1,1.000000,0.500292,1.000000,0.759610,True,True,True
ENSG00000225616,1.000000,0.500292,1.000000,0.759610,True,True,True
PTCHD1,1.000000,0.500292,1.000000,0.759610,True,True,True


In [8]:
excl_genes = gd.tag_bias_exclusion_genes(corr)

In [9]:
'HSPA1B' in excl_genes

False

In [10]:
adata_hvg = adata.copy()
adata_hvg

AnnData object with n_obs × n_vars = 4034 × 18833
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'chronopack', 'hvg', 'log1p', 'ot_graph', 'pca', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_traj_fates', 'terminal_fate_probs_traj_names', 'traj_clusters', 'traj_clusters_colors', 'traj_clusters_mass_retention_clusters_plotted', 'traj_clusters_mass

In [11]:
adata_hvg = adata.copy()
adata_hvg = filter_technical_genes(adata_hvg, exclude_genes=excl_genes)
adata_hvg = filter_low_count_genes(adata_hvg, min_total_counts=10, counts_layer="counts")
adata_hvg


AnnData object with n_obs × n_vars = 4034 × 11512
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'chronopack', 'hvg', 'log1p', 'ot_graph', 'pca', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_traj_fates', 'terminal_fate_probs_traj_names', 'traj_clusters', 'traj_clusters_colors', 'traj_clusters_mass_retention_clusters_plotted', 'traj_clusters_mass

In [12]:
print('PLAU' in set(adata_hvg.var.index)) # similar shape
print('CD83' in set(adata_hvg.var.index)) # similar shape
print('EGR1' in set(adata_hvg.var.index)) # similar shape
print('IER3' in set(adata_hvg.var.index)) # similar shape

True
True
True
True


In [13]:
# TPs
print('REL' in set(adata_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_hvg.var.index)) # similar shape

True
True
True
True
True
True


In [14]:
print(f"Genes retained after technical, time-tag-bias, and low-count filtering: {adata_hvg.n_vars}")
print("Low-count threshold applied before writing tradeSeq inputs: total counts >= 10")


Genes retained after technical, time-tag-bias, and low-count filtering: 11512
Low-count threshold applied before writing tradeSeq inputs: total counts >= 10


In [15]:
pd.Series(list(excl_genes)).to_csv('exclude_time-tag-bias_genes.csv', index=False, )
len(list(excl_genes))

816

In [16]:
adata_hvg.write_h5ad('../output/adata_prefiltered.1e4-variant.fixed.h5ad')
adata_full = adata.copy()

In [17]:
cp.tl.fit_gene_dynamics(
    adata_hvg,
    output_dir='../intmd/sc-K562-1e6-to-tradeSeq.fixed',
    full_adata=adata,
    prepare_inputs=True,
    run_backend=False,
)


GeneDynamicsResult(backend='tradeseq', key_added='gene_dynamics', metadata={'time_key': 'time', 'layer': 'counts', 'input_dir': '../intmd/sc-K562-1e6-to-tradeSeq.fixed', 'n_points': 9, 'workers': 4, 'screen_nknots': 4, 'final_nknots': 8, 'p_threshold': 0.05, 'outputs': {'association': 'asso_df.1trajectory.nknots4.csv', 'significant': 'sig_df.1trajectory.nknots4.csv', 'sce': 'sce.1trajectory.nknots8.rds', 'gamList': 'gamList.1trajectory.nknots8.rds'}, 'legacy_outputs': {'association': [], 'significant': [], 'sce': [], 'gamList': []}, 'prepared_paths': {'counts': '../intmd/sc-K562-1e6-to-tradeSeq.fixed/counts_genes_x_cells.csv.gz', 'cell_times': '../intmd/sc-K562-1e6-to-tradeSeq.fixed/cell_times.csv.gz', 'subset_genes': '../intmd/sc-K562-1e6-to-tradeSeq.fixed/subset_genes.csv', 'full_counts': '../intmd/sc-K562-1e6-to-tradeSeq.fixed/counts_genes_x_cells.full.csv.gz', 'full_cell_times': '../intmd/sc-K562-1e6-to-tradeSeq.fixed/cell_times.full.csv.gz', 'full_subset_genes': '../intmd/sc-K562-